In [1]:
%load_ext autoreload
%autoreload 2

import sys 
sys.path.append('/root/')

from dictionary_learning import ActivationBuffer
from nnsight import LanguageModel
from dictionary_learning import ActivationBuffer, AutoEncoder, AutoEncoderTopK
from dictionary_learning.trainers import StandardTrainer, TrainerTopK
from dictionary_learning.training import trainSAE

from datasets import load_dataset
import wandb
from tqdm import tqdm
import gc
import torch
from einops import rearrange
from collections import defaultdict
from termcolor import colored
import numpy as np
import itertools
from transformer_lens import HookedTransformer, utils

device = "cuda:0"

In [2]:
model_name = "gpt2" # can be any Huggingface model
model = LanguageModel(model_name, device_map=device)

In [3]:
dataset = load_dataset('Skylion007/openwebtext', split='train', streaming=True,
                                trust_remote_code=True)
class CustomData():
    def __init__(self, dataset):
        self.data = iter(dataset)

    def __iter__(self):
        return self

    def __next__(self):
        return next(self.data)['text']

data = CustomData(dataset)

In [8]:
# train the sparse autoencoder (SAE)
wandb.login()
activation_dim = 768

layer = 8
out_batch_size = 4096
tokens = 1_000_000
topk = 16
dictionary_size = 1 * activation_dim
token_mean_window = 64
ctx_len = 512
token_stride = 16

wandb_entity = 'jacobcd52'
wandb_project = 'token_mean_sae'
wandb_name = f"{model_name}_layer{layer}_k{topk}_dict{dictionary_size}_bs{out_batch_size}_window{token_mean_window}_stride{token_stride}"
save_dir = '/root/dictionary_learning/checkpoints'

submodule = model.transformer.h[layer]
trainer_cfg = {
    "trainer": TrainerTopK,
    "dict_class": AutoEncoderTopK,
    "activation_dim": activation_dim,
    "dict_size": dictionary_size,
    "device": device,
    "k": topk,
    "seed": 42,
    "wandb_name": wandb_name,
}

buffer = ActivationBuffer(
    data=data,
    model=model,
    submodule=submodule,
    d_submodule=activation_dim, # output dimension of the model component
    n_ctxs=1024,  # you can set this higher or lower depending on your available memory
    device=device,
    out_batch_size = 4096,
    refresh_batch_size = 256,
    io = 'out',
    token_mean_window = token_mean_window,
    token_stride = token_stride,
)  # buffer will yield batches of tensors of dimension = num_submodules * d_submodule

# wandb.init(entity=wandb_entity, project=wandb_project)
trainers = trainSAE(
    data=buffer,  # you could also use another (i.e. pytorch dataloader) here instead of buffer
    trainer_configs=[trainer_cfg],
    use_wandb = True,
    wandb_entity = wandb_entity,
    wandb_project = wandb_project,
    steps = int(tokens/out_batch_size),
    log_steps = 10,
    save_steps = 500,
    save_dir = save_dir,
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Refreshing activations:   0%|          | 0/131072 [11:52<?, ?it/s]

100%|██████████| 244/244 [11:56<00:00,  2.94s/it]


In [9]:
ae = trainers[0].ae

In [11]:
model = HookedTransformer.from_pretrained_no_processing('gpt2', device=device)

Loaded pretrained model gpt2 into HookedTransformer


In [23]:
strings = [next(data) for _ in range(2)]
tokens = model.tokenizer(strings, return_tensors='pt', padding=True, truncation=True, max_length=512)['input_ids'].to(device)

hook = "blocks.8.hook_resid_post"

gc.collect()
torch.cuda.empty_cache()

_, cache = model.run_with_cache(
    tokens,
    return_type="loss",
    names_filter = [hook]
)

act = cache[hook][:, 20:].mean(dim=1)
print(act.shape)
(ae(act) - act).pow(2).sum(-1).mean() / act.pow(2).sum(-1).mean()

torch.Size([2, 768])


tensor(0.0949, device='cuda:0', grad_fn=<DivBackward0>)

In [66]:
def get_batch_top_acts_inds(batch_tokens, hook_pt, k=None):
    # get model activations
    with torch.no_grad():
        _, cache = model.run_with_cache(
            batch_tokens,
            return_type="loss",
            names_filter=[hook_pt],
        )
        if k is not None:
            ae.k = k
        _, top_acts, top_inds = ae.encode(cache[hook_pt], return_topk=True)
    return top_acts, top_inds

def get_all_top_acts_inds(tokens, hook_list, num_contexts=1_000, batch_size=64, k=None):
    top_acts = []
    top_inds = []
    for b in tqdm(range(num_contexts // batch_size)):
        batch_tokens = tokens[b*batch_size:(b+1)*batch_size]
        batch_top_acts, batch_top_inds = get_batch_top_acts_inds(batch_tokens, hook_list, k=k)
        top_acts.append(batch_top_acts.cpu())
        top_inds.append(batch_top_inds.cpu())

    top_acts = torch.cat(top_acts, dim=0)
    top_inds = torch.cat(top_inds, dim=0)
    return top_acts, top_inds


def get_top_windows(acts: torch.Tensor, inds: torch.Tensor, feature_id: int, window_length: int = 64):
    assert acts.shape == inds.shape, "acts and inds must have the same shape"
    
    mask = (inds == feature_id)
    masked_acts = torch.where(mask, acts, torch.tensor(0.0))
    
    batch_size, n_ctx, k = acts.shape
    
    # Sum across the k dimension
    batch_acts = masked_acts.sum(dim=-1)
    
    # Use unfold to create sliding windows
    windows = batch_acts.unfold(1, window_length, 1)
    window_sums = windows.sum(dim=-1)
    
    # Get the indices of top windows
    flat_sums = window_sums.view(-1)
    top_values, top_indices = torch.topk(flat_sums, k=min(batch_size * (n_ctx - window_length + 1), 1000))
    
    # Convert flat indices to (batch, start) indices
    batch_indices = top_indices // (n_ctx - window_length + 1)
    start_indices = top_indices % (n_ctx - window_length + 1)
    
    windows = [(b.item(), s.item(), s.item() + window_length, v.item() / window_length) 
               for b, s, v in zip(batch_indices, start_indices, top_values)]
    
    return windows

def merge_windows(windows):
    merged = []
    for batch, windows_in_batch in itertools.groupby(sorted(windows), key=lambda x: x[0]):
        current_window = None
        for window in windows_in_batch:
            if current_window is None:
                current_window = list(window)
            elif window[1] <= current_window[2]:  # Windows overlap
                # Extend the end of the current window if necessary
                current_window[2] = max(current_window[2], window[2])
                # Update the average activation (weighted by window lengths)
                total_activation = (current_window[3] * (current_window[2] - current_window[1]) + 
                                    window[3] * (window[2] - window[1]))
                total_length = current_window[2] - current_window[1]
                current_window[3] = total_activation / total_length
            else:
                merged.append(tuple(current_window))
                current_window = list(window)
        if current_window:
            merged.append(tuple(current_window))
    return merged

def color_gradient(val, min_val, max_val):
    norm_val = (val - min_val) / (max_val - min_val)
    # Create a gradient from dark blue (0, 0, 50) to bright red (255, 0, 0)
    r = int(255 * norm_val) + 40
    g = 10
    b = 10 #int(100 * (1 - norm_val))
    return f"\033[48;2;{r};{g};{b}m\033[97m"  # White text on colored background

def print_top_windows(tokens, acts, inds, feature_id, window_length: int = 64, num_windows_to_show: int = 25):
    windows = get_top_windows(acts, inds, feature_id, window_length)
    merged_windows = merge_windows(windows)
    
    batch_size, n_ctx, k = acts.shape
    
    print(f"Feature {feature_id} top windows:\n")
    
    for i, (batch, start, end, avg_activation) in enumerate(merged_windows):
        if i >= num_windows_to_show:
            break
        
        if batch < 0 or batch >= batch_size:
            continue
        
        # Extract the sequence
        sequence = tokens[batch, start:end].tolist()
        
        # Get activations for this window
        window_acts = acts[batch, start:end]
        window_inds = inds[batch, start:end]
        
        # Create a list of (token, activation) pairs
        token_acts = []
        for t, (act, ind) in enumerate(zip(window_acts, window_inds)):
            token = model.tokenizer.decode([sequence[t]])
            if feature_id in ind:
                act_value = act[ind == feature_id].max().item()
            else:
                act_value = 0
            token_acts.append((token, act_value))
        
        # Get min and max activation values for color scaling
        act_values = [act for _, act in token_acts if act > 0]
        if act_values:
            min_act, max_act = min(act_values), max(act_values)
        else:
            min_act, max_act = 0, 1  # Fallback if no activations
        
        # Print the sequence with color-graded highlighting
        print(f"Batch {batch}, Window {start}-{end} (average activation: {avg_activation:.4f}):")
        for token, act_value in token_acts:
            if act_value > 0:
                color = color_gradient(act_value, min_act, max_act)
                print(f"{color}{token}\033[0m", end='')
            else:
                print(f"\033[10m{token}\033[0m", end='')  # Dark gray for non-activated tokens
        print("\n" + "-"*50 + "\n")

# Example usage:
# print_top_windows(tokens, acts, inds, feature_id, window_length=64, num_windows_to_show=25)

def print_top_logits(feature_id, num_to_show=10):
    if isinstance(feature_id, torch.Tensor):
        feature_id = int(feature_id.item())
    dec = ae.decoder.weight[:, feature_id]


    logits = dec @ model.W_U

    top_logits, top_inds = logits.topk(k=num_to_show)

    print(f"Feature {feature_id} top logits:\n")
    for i in range(num_to_show):
        print(f"{top_logits[i].item():.1f}:  {model.tokenizer.decode(top_inds[i].item())}")


TODO:

- print feature frequency
- cluster features
- train wider and with higher k
- evaluate loss with patching
- some dashboards not making sense (e.g. no highlighted tokens, but avg activation is supposedly high)
- hover over to show activation

In [129]:
eval_data = load_dataset('stas/openwebtext-10k', split='train').shuffle(42)
tokenized_data = utils.tokenize_and_concatenate(eval_data, model.tokenizer, max_length=512)
eval_tokens = tokenized_data["tokens"]
eval_tokens.shape

Map (num_proc=10):   0%|          | 0/10000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (55651 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (58538 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (59962 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (55565 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (63725 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence

torch.Size([21997, 512])

In [56]:
gc.collect()
torch.cuda.empty_cache()
top_acts, top_inds = get_all_top_acts_inds(eval_tokens, 'blocks.8.hook_resid_post', num_contexts=10_000, batch_size=64, k=16)

  1%|▏         | 2/156 [00:02<03:43,  1.45s/it]


KeyboardInterrupt: 

In [144]:
feature_id = 18
print_top_windows(eval_tokens, top_acts, top_inds, feature_id, window_length=64, num_windows_to_show=20)
print_top_logits(feature_id, num_to_show=10)

Feature 18 top windows:

Batch 1675, Window 36-147 (average activation: 36.6699):
 all, Key kept writing.

Back with a newfound fire, the group—also including guitarist Ryan Mendez and bassist Josh Portman—crafted Lift A Sail, their boldest, most exploratory work to date, fusing their familiar West Coast-inspired pop-punk with a wide palette of previously unused textures including heavy splashes of ��90s alt rock and electronica. Intrigued, we caught up with Key to find out how Yellowcard arrived at their radical new sound.

Before you dive into our interview with
--------------------------------------------------

Batch 2140, Window 395-512 (average activation: 49.8837):
 According to WDI, the lounge was once a secret stomping ground for famous touring magicians, local boardwalk illusionists and the loveliest magician��s assistants of the day. After every magic show, these prestidigitators would gather at the bar, where they��d conjure up new cocktails, swap magic tips and tricks, and

In [117]:
prompts = ["The caucus came to prominence in 1976 when Jimmy Carter used Iowa as a springboard", 
           " So if you get pregnant, oh well. You have a baby. Great. I've been married 23 years."]

model.tokenizer.padding_side = "left"
tokens = torch.stack(
    [
    model.tokenizer.encode(prompt, return_tensors='pt', padding=True, truncation=True, max_length=10).to(device).squeeze()
    for prompt in prompts
    ]
)
print(tokens.shape)
output = model.generate(tokens, max_new_tokens=512)

torch.Size([2, 10])


  0%|          | 0/512 [00:00<?, ?it/s]

In [133]:
for string in model.tokenizer.batch_decode(output):
    print("----------------------")
    print(string)

----------------------
The caucus came to prominence in 1976 when Jimmy Carter was elected the first African-American president. He received 39 of the top 99 delegates, mostly members of the Democratic Partys.

1951: Karl Ben-Gurion was elected First Vice President off the ballot. When told first he was afraid of Clinton, then turned back to a campaign promise:

"Now then, in consequence of those experiences we know, we must define, let's define decency." ~Karl Ben-Gurion, writing to his wife, through a translator

Hay was Jewish. Enter Jane Harvey. On May 19, 1950, with, then in her 30s. was born of Jewish parents, Betanic Ashkenazi Ezra and Sonu Nahum Shachar. Let's see Ben-Gurion, 76, see what he saw. His wife already was college educated, an obvious indication of the Jewish identity, but "I live by, " kissed a Jewish journalist Jacob Fiinah — one of American Jewish Childhood New York

1953: Henry VIII attempted to build Cairo more dangerously after Egyptian workers were turned agai

In [119]:
with torch.no_grad():
    _, cache = model.run_with_cache(
    output,
    return_type="loss",
    names_filter = [hook]
    )   

act = cache[hook]
print(act.shape)

def moving_average(acts, window_size=64, stride_length=16):
    batch, num_tokens, d = acts.shape
    
    # Calculate the number of windows
    num_windows = (num_tokens - window_size) // stride_length + 1
    
    # Create a tensor to store the results
    result = torch.zeros(batch, num_windows, d, device=acts.device)
    
    for i in range(num_windows):
        start = i * stride_length
        end = start + window_size
        result[:, i, :] = acts[:, start:end, :].mean(dim=1)
    
    return result

moving_average(act).shape


torch.Size([2, 522, 768])


torch.Size([2, 29, 768])

In [123]:
_, topk_acts, topk_inds = ae.encode(moving_average(act), return_topk=True)
topk_inds[1]
